In [1]:
%reset -f
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split

import joblib
import json
from pathlib import Path

import sys
#sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
import importlib
#importlib.reload(eva)
#importlib.reload(prep)


In [3]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies
task_num = 3 + 2 * NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: fluctuation

# The folder storing datasets splited latter.
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20"
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20_fl"
folder_path = "../../Databases/Ver1.9/ver1.9.6.1_upto20"

for rep in range(1,2):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_fl_v1/split_fl_{rep}.csv")
    #split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")
    split_df = pd.read_csv(f"split_ver1.9.6.1_upto20/split_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        sample_size=70000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        sample_size=15000,
        random_state=42
    )

    train_dataset = acht.CommunityDataset(Xg_tr, XA_tr, Xd_tr, y_tr)
    test_dataset = acht.CommunityDataset(Xg_te, XA_te, Xd_te, y_te)

    n_val = int(len(train_dataset)*0.2)
    n_train = len(train_dataset)-n_val
    train_ds, val_ds = random_split(train_dataset, [n_train,n_val])

    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    model = acht.MDTrans2(NumSpecies).to(device) # Select deep learning architecture.
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0], device=device)) # positive class weight = 1
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    early_stop = acht.EarlyStopping(patience=8, min_delta=1e-4)
    history = {"loss":[], "val_loss":[]}

    best_state = None

    for epoch in range(140):
        model.train()
        train_loss = 0
        
        for Xg,XA,Xd,y in train_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(Xg,XA,Xd)

            loss = criterion(logits,y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        model.eval()

        val_loss = 0

        with torch.no_grad():

            for Xg,XA,Xd,y in val_loader:

                Xg = Xg.to(device)
                XA = XA.to(device)
                Xd = Xd.to(device)
                y = y.to(device)

                logits = model(Xg,XA,Xd)
                loss = criterion(logits,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"train={train_loss:.5f} "
            f"val={val_loss:.5f}"
        )

        if val_loss < early_stop.best_loss:
            best_state = model.state_dict()

        if early_stop.step(val_loss):
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    # Model saving
    #save_dir = Path(f"/work/home/acychkqsp6/zrm/MasterThesis/saved_models/fluctuation_15fold/sp3to20_Fl_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/fluctuation_3fold_v0/sp10_Fl_transpt3_r{rep}")
    save_dir = Path(f"../../saved_models/fluctuation_v2/sp10_Fl_transpt3_r{rep}")

    save_dir.mkdir(parents=True, exist_ok=True)  # auto-create directories
    torch.save(model.state_dict(), save_dir / "model.pt")
    joblib.dump(scaler_g, save_dir / "scaler_g.pkl")
    joblib.dump(scaler_d, save_dir / "scaler_d.pkl")

    config = {
        "NumSpecies": NumSpecies,
        "featgam": featgam,
        "task_num": task_num,
        "embed_dim": 64,
        "num_heads": 4,
        "ff_dim": 128
    }

    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f)

    model.eval()
    preds = []

    with torch.no_grad():

        for Xg, XA, Xd, _ in test_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)

            logits = model(Xg,XA,Xd)
            prob = torch.sigmoid(logits)
            preds.append(prob.cpu().numpy())

    y_prob = np.concatenate(preds).ravel()

    threshold=0.8
    y_pred = (y_prob >= threshold).astype(int)

    metrics = eva.eval_binary(y_te, y_prob, threshold=threshold)
    df_metrics = pd.DataFrame([metrics])
    print(df_metrics.to_string())
    

tr shape = (80853, 465)
te shape = (20213, 465)
Epoch 1 train=0.45901 val=0.41985
Epoch 2 train=0.39686 val=0.39029
Epoch 3 train=0.38006 val=0.38602
Epoch 4 train=0.37655 val=0.38625
Epoch 5 train=0.37371 val=0.38180
Epoch 6 train=0.37029 val=0.37779
Epoch 7 train=0.37143 val=0.38167
Epoch 8 train=0.37049 val=0.37890
Epoch 9 train=0.36720 val=0.37927
Epoch 10 train=0.36610 val=0.37841
Epoch 11 train=0.36426 val=0.37851
Epoch 12 train=0.36322 val=0.38178
Epoch 13 train=0.36336 val=0.38193
Epoch 14 train=0.36129 val=0.38023
Early stopping
   Accuracy  Balanced_Acc   F1  MCC      AUC     AUPRC  AUPRC_rel     Brier  pos_rate
0  0.849867           0.5  0.0  0.0  0.70413  0.246875    0.64437  0.119722  0.150133
